# Regression Simple Example For Cloud Function

## Downloading Library

In [ ]:
!python --version

Python 3.10.12


In [ ]:
!pip install catboost

## CatBoost Example

### Importing Library

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from catboost import CatBoostRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error

import pickle

### Reading The Dataset

In [ ]:
df = pd.read_csv('/content/50_Startups.csv')
df = pd.get_dummies(df, drop_first=True)

In [ ]:
df.head()

,R&D Spend,Administration,Marketing Spend,Profit,State_Florida,State_New York
0,165349.20,136897.80,471784.10,192261.83,False,True
1,162597.70,151377.59,443898.53,191792.06,False,False
2,153441.51,101145.55,407934.54,191050.39,True,False
3,144372.41,118671.85,383199.62,182901.99,False,True
4,142107.34,91391.77,366168.42,166187.94,True,False


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   R&D Spend        50 non-null     float64
 1   Administration   50 non-null     float64
 2   Marketing Spend  50 non-null     float64
 3   Profit           50 non-null     float64
 4   State_Florida    50 non-null     bool   
 5   State_New York   50 non-null     bool   
dtypes: bool(2), float64(4)
memory usage: 1.8 KB


In [ ]:
df.isnull().sum()

,0
R&D Spend,0
Administration,0
Marketing Spend,0
Profit,0
State_Florida,0
State_New York,0


### Splitting Independent & Dependent Variable

In [ ]:
X = df.drop('Profit', axis=1)
y = df['Profit']

In [ ]:
X

,R&D Spend,Administration,Marketing Spend,State_Florida,State_New York
0,165349.20,136897.80,471784.10,False,True
1,162597.70,151377.59,443898.53,False,False
2,153441.51,101145.55,407934.54,True,False
3,144372.41,118671.85,383199.62,False,True
4,142107.34,91391.77,366168.42,True,False
5,131876.90,99814.71,362861.36,False,True
6,134615.46,147198.87,127716.82,False,False
7,130298.13,145530.06,323876.68,True,False
8,120542.52,148718.95,311613.29,False,True
9,123334.88,108679.17,304981.62,False,False


In [ ]:
y

,Profit
0,192261.83
1,191792.06
2,191050.39
3,182901.99
4,166187.94
5,156991.12
6,156122.51
7,155752.60
8,152211.77
9,149759.96


### Splitting Into Train & Test

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

### Defining Model

In [ ]:
catboost_model = CatBoostRegressor(iterations=500, learning_rate=0.05, depth=6, verbose=0)

### Train The Model

In [ ]:
catboost_model.fit(X_train, y_train)

### Predicting The Test Set

In [ ]:
y_pred = catboost_model.predict(X_test)

### Evaluating Model Performance

In [ ]:
rmse_catboost = np.sqrt(root_mean_squared_error(y_test, y_pred))

print(f"CatBoost Model RMSE: {rmse_catboost}")

CatBoost Model RMSE: 110.0833995939618


### Creating Function For Predicting New Data

In [ ]:
def predict_catboost(input_data):
    # Convert input data to DataFrame
    input_df = pd.DataFrame([input_data], columns=X.columns)

    # Make prediction
    predicted_profit = catboost_model.predict(input_df)

    return predicted_profit[0]

# Example input
input_data = [0, 1, 120000, 0, 1]
predicted_profit_catboost = predict_catboost(input_data)

print(f"Predicted Profit (CatBoost): ${predicted_profit_catboost}")

Predicted Profit (CatBoost): $68875.76418625368


### Saving The Model

In [ ]:
with open('catboost_model.pkl', 'wb') as file:
    pickle.dump(catboost_model, file)

print("CatBoost model saved!")

CatBoost model saved!


### Load Model and Predict Data

In [ ]:
# Load the saved CatBoost model from file
with open('catboost_model.pkl', 'rb') as file:
    loaded_catboost_model = pickle.load(file)

# Function to make predictions with the loaded model
def predict_catboost(input_data):
    input_df = pd.DataFrame([input_data], columns=X.columns)
    predicted_profit = loaded_catboost_model.predict(input_df)
    return predicted_profit[0]

# Example input for prediction
input_data = [0, 1, 120000, 0, 1]  # Example input
predicted_profit_catboost = predict_catboost(input_data)
print(f"Predicted Profit (CatBoost): ${predicted_profit_catboost}")

Predicted Profit (CatBoost): $68875.76418625368


## Tensorflow Example

### Import Library

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Adam

from sklearn.preprocessing import StandardScaler

### Reading The Dataset

In [ ]:
df = pd.read_csv('/content/50_Startups.csv')
df = pd.get_dummies(df, drop_first=True)

## Splitting The Dependent & Independent Variable

In [ ]:
X = df.drop('Profit', axis=1)
y = df['Profit']

### Scale The Dataset

In [ ]:
scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

### Defining Neural Network Model

In [ ]:
nn_model = Sequential([
    Dense(64, activation='relu', input_dim=X_scaled.shape[1]),
    Dense(32, activation='relu'),
    Dense(1)
])
nn_model.compile(optimizer=Adam(learning_rate=0.001), loss='mean_squared_error')
nn_model.fit(X_scaled, y, epochs=100, batch_size=32, validation_split=0.2, verbose=0)

/usr/local/lib/python3.10/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


### Saving The Neural Network and Scaler

In [ ]:
nn_model.save('neural_network_model.h5')

with open('scaler.pkl', 'wb') as file:
    pickle.dump(scaler, file)

### Load The Neural Network and Scaler

In [ ]:
loaded_nn_model = tf.keras.models.load_model('neural_network_model.h5')

with open('scaler.pkl', 'rb') as file:
    loaded_scaler = pickle.load(file)

### Predict New Data Using Loaded Model and Scaler

In [ ]:
def predict_nn(input_data):
    # Scale the input data using the loaded scaler
    input_df = pd.DataFrame([input_data], columns=X.columns)
    input_scaled = loaded_scaler.transform(input_df)

    # Make prediction
    predicted_profit = loaded_nn_model.predict(input_scaled)
    return predicted_profit[0][0]

# Example input for prediction
input_data = [0, 1, 120000, 0, 1]  # Example input
predicted_profit_nn = predict_nn(input_data)
print(f"Predicted Profit (Neural Network): ${predicted_profit_nn}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 72ms/step
Predicted Profit (Neural Network): $58.13029861450195


## Expected Format (CatBoost)

### Code

In [ ]:
import os
import pickle
import numpy as np
import pandas as pd
from google.cloud import storage
from catboost import CatBoostRegressor

# Global model variable
model = None

# Download model file from cloud storage bucket
def download_model_file():
    BUCKET_NAME = "YOUR_MODEL_BUCKET_NAME"
    PROJECT_ID = "YOUR_GCP_PROJECT_ID"
    GCS_MODEL_FILE = "catboost_model.pkl"

    # Initialise a client
    client = storage.Client(PROJECT_ID)

    # Create a bucket object for our bucket
    bucket = client.get_bucket(BUCKET_NAME)

    # Download the CatBoost model
    catboost_blob = bucket.blob(GCS_MODEL_FILE)
    catboost_blob.download_to_filename('/tmp/catboost_model.pkl')

# Main entry point for the cloud function
def catboost_predict(request):
    global model

    if not model:
        download_model_file()
        model = pickle.load(open("/tmp/catboost_model.pkl", 'rb'))

    # Get the features sent for prediction
    params = request.get_json()

    if params and 'features' in params:
        # Create a DataFrame from the input features
        input_df = pd.DataFrame([params['features']], columns=['R&D Spend', 'Administration', 'Marketing Spend', 'State_Florida', 'State_New York'])

        # Make prediction
        predicted_profit = model.predict(input_df)
        return {"predicted_profit": predicted_profit[0]}

    else:
        return {"error": "No features provided for prediction"}

### Requirements File

In [ ]:
# catboost==1.2  # Versi dapat disesuaikan sesuai kebutuhan
# pandas==1.5.3  # Versi dapat disesuaikan sesuai kebutuhan
# numpy==1.24.2  # Versi dapat disesuaikan sesuai kebutuhan
# google-cloud-storage==2.10.0  # Untuk mengakses Google Cloud Storage

## Expected Format (Tensorflow)

### Code

In [ ]:
import os
import pickle
import numpy as np
import pandas as pd
from google.cloud import storage
import tensorflow as tf

# Global model variable
model = None
scaler = None

# Download model file from cloud storage bucket
def download_model_file():
    BUCKET_NAME        = "YOUR_MODEL_BUCKET_NAME"
    PROJECT_ID         = "YOUR_GCP_PROJECT_ID"
    GCS_MODEL_FILE     = "neural_network_model.h5"
    GCS_SCALER_FILE    = "scaler.pkl"

    # Initialise a client
    client = storage.Client(PROJECT_ID)

    # Create a bucket object for our bucket
    bucket = client.get_bucket(BUCKET_NAME)

    # Download the Neural Network model
    nn_blob = bucket.blob(GCS_MODEL_FILE)
    nn_blob.download_to_filename('/tmp/neural_network_model.h5')

    # Download the scaler
    scaler_blob = bucket.blob(GCS_SCALER_FILE)
    scaler_blob.download_to_filename('/tmp/scaler.pkl')

# Main entry point for the cloud function
def nn_predict(request):
    global model, scaler

    if not model:
        download_model_file()
        model = tf.keras.models.load_model('/tmp/neural_network_model.h5')
        scaler = pickle.load(open('/tmp/scaler.pkl', 'rb'))

    # Get the features sent for prediction
    params = request.get_json()

    if params and 'features' in params:
        # Scale the input data
        input_df = pd.DataFrame([params['features']], columns=['R&D Spend', 'Administration', 'Marketing Spend', 'State_Florida', 'State_New York'])
        input_scaled = scaler.transform(input_df)

        # Make prediction
        predicted_profit = model.predict(input_scaled)
        return {"predicted_profit": predicted_profit[0][0]}

    else:
        return {"error": "No features provided for prediction"}

### Requirements File

In [ ]:
# tensorflow==2.13.0  # Versi dapat disesuaikan sesuai kebutuhan
# pandas==1.5.3  # Versi dapat disesuaikan sesuai kebutuhan
# numpy==1.24.2  # Versi dapat disesuaikan sesuai kebutuhan
# google-cloud-storage==2.10.0  # Untuk mengakses Google Cloud Storage